In [32]:
import importlib
import utilities
import shutil
import sys
import os
importlib.reload(utilities)
from utilities import get_first_file, rename_files
import requests
import argparse

from dotenv import load_dotenv
load_dotenv(override=True) 

PAGE_ID = os.environ['FB_MATH_KIDS_PAGE_ID']
PAGE_ACCESS_TOKEN = os.environ['FB_MATH_KIDS_TOKEN']

GRAPH_URL = f"https://graph.facebook.com/v19.0/{PAGE_ID}/videos"

CHUNK_SIZE = 4 * 1024 * 1024  # 4MB chunks

In [33]:
VIDEO_FOLDER = '/Users/sangdo/Downloads/math_games_video/output/fb_video/'   #contain mp4 files

DESCRIPTION = """
300+ games as PDF file in the first comment.
Math games for your kids at the spare time - Puzzle {index}

We introduce a range of various games:
Addition matrix
Hidden gems
Word search
Crossword numbers
Balance game
Find lines
Triangle sum
Balance fruit
Object coordination
Spy game
Detect shape
Bee house
"""

In [34]:
def get_post_id_from_video_id(video_id):
    url = f"https://graph.facebook.com/v19.0/{video_id}"
    
    params = {
        "fields": "post_id",
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json().get("post_id")  # e.g. "123456789_987654321"

In [ ]:
def start_upload(file_size):
    params = {
        "upload_phase": "start",
        "file_size": file_size,
        "access_token": PAGE_ACCESS_TOKEN
    }

    r = requests.post(GRAPH_URL, data=params, timeout=60)
    r.raise_for_status()
    return r.json()


def transfer_chunks(upload_session_id, start_offset, video_file):
    while True:
        start = int(start_offset)
        video_file.seek(start)
        chunk = video_file.read(CHUNK_SIZE)

        files = {
            "video_file_chunk": chunk
        }

        data = {
            "upload_phase": "transfer",
            "upload_session_id": upload_session_id,
            "start_offset": start_offset,
            "access_token": PAGE_ACCESS_TOKEN
        }

        r = requests.post(GRAPH_URL, data=data, files=files, timeout=300)
        r.raise_for_status()
        result = r.json()

        start_offset = result["start_offset"]
        end_offset = result["end_offset"]

        print(f"Uploaded bytes {start_offset} / {end_offset}")

        if start_offset == end_offset:
            break

    return start_offset

#uploaded video, now creating post
def finish_upload(upload_session_id, index):
    description = DESCRIPTION.replace('{index}', str(index))
    data = {
        "upload_phase": "finish",
        "upload_session_id": upload_session_id,
        "description": description,
        "published": "true",
        "access_token": PAGE_ACCESS_TOKEN
    }

    r = requests.post(GRAPH_URL, data=data, timeout=60)
    r.raise_for_status()
    print('Result after posting to FB page: ', r.json())
    return r.json()


def upload_video():
    video_path, index = get_first_file(VIDEO_FOLDER, 'mp4')

    file_size = os.path.getsize(video_path)

    print("Starting upload session...")
    start = start_upload(file_size)

    upload_session_id = start["upload_session_id"]
    start_offset = start["start_offset"]

    with open(video_path, "rb") as f:
        transfer_chunks(upload_session_id, start_offset, f)

    print("Finishing upload... " + str(index))
    finish = finish_upload(upload_session_id, index)
    #there is no video id returned

    # video_id = finish["video_id"]
    # print("Video uploaded successfully!")
    # print("Video ID:", video_id)
    # post_id = get_post_id(video_id)
    # print("Post ID:", post_id)
    # #rename uploaded file with different extension (for comment later)
    # os.rename(video_path, video_path.replace('.mp4', '.fb'))

    return finish


if __name__ == "__main__":
    post_id = upload_video()

Starting upload session...
Uploaded bytes 1030029 / 1030029
Finishing upload... 19
Result after uploading video:  {'success': True}
Returned result of post: {'success': True}


In [36]:
def get_latest_video_id():
    url = f"https://graph.facebook.com/v19.0/{PAGE_ID}/videos"
    
    params = {
        "fields": "id,post_id",
        "limit": 1,                    # Only get the latest one
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    
    videos = r.json().get("data", [])
    if not videos:
        raise Exception("No videos found")
    print(videos)
    
    return videos[0]["id"], videos[0].get("post_id")

# get_latest_video_id()

In [37]:
def get_video_detail(video_id):
    url = f"https://graph.facebook.com/v19.0/{video_id}"
    
    params = {
        "fields": "id,title,description,length,place,status,created_time,thumbnails,permalink_url",
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()
#
# video_detail = get_video_detail('1690219669060171')
# print(video_detail)

In [38]:
def delete_post(post_id):
    url = f"https://graph.facebook.com/v19.0/{post_id}"
    
    params = {
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.delete(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()  # returns {"success": true}
#
# delete_post('355786257625985_122220911462455420')

In [39]:
# post_id = get_post_id_from_video_id('1690219669060171')
# print("Post ID:", post_id)

In [ ]:
def get_first_comment(post_id):
    url = f"https://graph.facebook.com/v20.0/{post_id}/comments"
    
    params = {
        "fields": "id,message,created_time,from",
        "limit": 1,                    # Only fetch the first comment
        "order": "chronological",      # Oldest first
        "access_token": PAGE_ACCESS_TOKEN
    }
    
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    
    data = r.json().get("data", [])
    if not data:
        return None  # No comments yet
    
    return data[0]
#
first_comment = get_first_comment('355786257625985_122220929750455420') #only post from API can read comment
print(first_comment)


None


In [41]:
def check_token_permissions():
    url = "https://graph.facebook.com/v20.0/me"
    
    params = {"access_token": PAGE_ACCESS_TOKEN}
    
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

# Check what permissions your token actually has
# perms = check_token_permissions()
# print(perms)
